# Decision Trees for Highly Cited Research Papers

This notebook predicts whether a research paper is highly cited instead of whether a song is popular.

## Setup

In [ ]:
import json
import sqlite3
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import polars as pl
from scipy.sparse import csr_matrix, hstack
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier, plot_tree

## Load Research Paper Data

In [ ]:
# Load the joined OpenAlex + Semantic Scholar data used in the K-Means notebook.
DB_PATH = Path("papers.db")
if not DB_PATH.exists():
    DB_PATH = Path("../papers.db")

JOINED_QUERY = """
SELECT
    o.openalex_id,
    o.doi,
    o.doi_normalized,
    o.title,
    o.publication_year,
    o.cited_by_count,
    o.author_count,
    o.primary_topic,
    o.primary_subfield,
    o.primary_field,
    o.primary_domain,
    s.abstract_text,
    s.abstract_tfidf_vector
FROM openalex_papers AS o
JOIN semanticscholar_papers AS s
    ON o.doi_normalized = s.doi_normalized
WHERE s.abstract_tfidf_vector IS NOT NULL
  AND TRIM(s.abstract_tfidf_vector) <> ''
  AND o.cited_by_count IS NOT NULL
"""

with sqlite3.connect(DB_PATH) as conn:
    df = pl.read_database(query=JOINED_QUERY, connection=conn)

print(f"Joined rows with abstract vectors and citation counts: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")
df.head()

## Binarize Citation Popularity

The homework used `binarize_popularity` to mark songs as popular. Here, the same function name creates a `highly_cited` boolean label from `cited_by_count`.

In [ ]:
def binarize_popularity(df: pl.DataFrame, threshold: int = 1) -> pl.DataFrame:
    """
    Add a 'highly_cited' boolean column indicating whether a paper is highly cited.

    Requirements:
    - Add a new column 'highly_cited' (boolean):
        - True if cited_by_count >= threshold
        - False if cited_by_count < threshold
    - Do not modify any existing columns

    Args:
        df: Polars DataFrame with a 'cited_by_count' column
        threshold: citation-count cutoff

    Returns:
        DataFrame with added 'highly_cited' boolean column

    Example:
        >>> sample = pl.DataFrame({'cited_by_count': [0, 1, 10]})
        >>> result = binarize_popularity(sample, threshold=1)
        >>> result['highly_cited'].to_list()
        [False, True, True]
    """
    return df.with_columns(
        (pl.col("cited_by_count").fill_null(0) >= threshold).alias("highly_cited")
    )


# Example test:
sample = pl.DataFrame({"cited_by_count": [0, 1, 10]})
binarize_popularity(sample, threshold=1)["highly_cited"].to_list()
# Expected: [False, True, True]

In [ ]:
# The citation distribution is very skewed, so use the 95th percentile as a data-aware cutoff.
# For this dataset, that cutoff is 1 citation. Raise this manually for a stricter definition.
CITATION_THRESHOLD = max(
    1,
    int(df.select(pl.col("cited_by_count").quantile(0.95, interpolation="nearest")).item()),
)

df_labeled = binarize_popularity(df, threshold=CITATION_THRESHOLD)

label_counts = (
    df_labeled
    .group_by("highly_cited")
    .len()
    .with_columns((pl.col("len") / df_labeled.height).alias("fraction"))
    .sort("highly_cited")
)

print(f"Highly cited threshold: cited_by_count >= {CITATION_THRESHOLD}")
label_counts

## Prepare Features

`cited_by_count` is the target source, so it is intentionally excluded from the feature matrix. The notebook reuses the stored abstract TF-IDF vectors from `papers.db` instead of fitting a new text vectorizer here.

In [ ]:
NUMERIC_FEATURES = ["publication_year", "log_author_count"]
CATEGORICAL_FEATURES = ["primary_topic", "primary_subfield", "primary_field", "primary_domain"]

# Create the binary target once, then split row indices before fitting any supervised model.
# The categorical encoder below is still fit only on training rows.
y_binary = df_labeled["highly_cited"].to_numpy().astype(int)
train_indices, test_indices = train_test_split(
    np.arange(df_labeled.height),
    test_size=0.20,
    random_state=42,
    stratify=y_binary,
)

df_train = df_labeled[train_indices]
df_test = df_labeled[test_indices]
y_train = y_binary[train_indices]
y_test = y_binary[test_indices]

print("Train rows:", df_train.height)
print("Test rows:", df_test.height)
print("Train positives:", int(y_train.sum()))
print("Test positives:", int(y_test.sum()))

In [ ]:
def build_numeric_matrix(papers_df: pl.DataFrame) -> np.ndarray:
    """Build the numeric feature block without using full-dataset statistics."""
    return papers_df.select([
        # Direct fill/value: all scraped papers are from 2026.
        pl.lit(2026.0).alias("publication_year"),
        pl.col("author_count")
          .fill_null(0)
          .clip(lower_bound=0)
          .log1p()
          .cast(pl.Float64)
          .alias("log_author_count"),
    ]).to_numpy()


numeric_train = build_numeric_matrix(df_train)
numeric_test = build_numeric_matrix(df_test)

print("Numeric train shape:", numeric_train.shape)
print("Numeric test shape:", numeric_test.shape)

In [ ]:
def build_categorical_values(papers_df: pl.DataFrame) -> np.ndarray:
    """Return categorical columns as a NumPy array for OneHotEncoder."""
    return papers_df.select([
        pl.col(column).fill_null("Unknown") for column in CATEGORICAL_FEATURES
    ]).to_numpy()


# Fit on training data only; transform the test data with the fitted encoder.
categorical_encoder = OneHotEncoder(
    handle_unknown="ignore",
    min_frequency=20,
    sparse_output=True,
)
categorical_train = categorical_encoder.fit_transform(build_categorical_values(df_train))
categorical_test = categorical_encoder.transform(build_categorical_values(df_test))

print("Categorical train shape:", categorical_train.shape)
print("Categorical test shape:", categorical_test.shape)

In [ ]:
# Use the abstract TF-IDF vectors already stored in papers.db.
# These vectors are JSON objects with dimension, indices, and values fields.
def stored_tfidf_vectors_to_csr(vector_jsons: list[str]) -> csr_matrix:
    """Convert stored TF-IDF JSON vectors from the database into a CSR matrix."""
    data = []
    indices = []
    indptr = [0]
    dimension = None

    for vector_json in vector_jsons:
        vector = json.loads(vector_json)
        vector_dimension = int(vector["dimension"])

        if dimension is None:
            dimension = vector_dimension
        elif dimension != vector_dimension:
            raise ValueError("Stored abstract TF-IDF vectors have inconsistent dimensions")

        indices.extend(vector["indices"])
        data.extend(vector["values"])
        indptr.append(len(indices))

    return csr_matrix((data, indices, indptr), shape=(len(vector_jsons), dimension))


abstract_matrix = stored_tfidf_vectors_to_csr(df_labeled["abstract_tfidf_vector"].to_list())
abstract_train = abstract_matrix[train_indices]
abstract_test = abstract_matrix[test_indices]

# The stored vectors do not include the original vocabulary, so use dimension names.
abstract_feature_names = [
    f"abstract_tfidf_{i}" for i in range(abstract_matrix.shape[1])
]

print("Stored abstract TF-IDF shape:", abstract_matrix.shape)
print("Abstract TF-IDF train shape:", abstract_train.shape)
print("Abstract TF-IDF test shape:", abstract_test.shape)

In [ ]:
X_train = hstack([
    csr_matrix(numeric_train),
    categorical_train,
    abstract_train,
], format="csr")

X_test = hstack([
    csr_matrix(numeric_test),
    categorical_test,
    abstract_test,
], format="csr")

feature_names = (
    NUMERIC_FEATURES
    + categorical_encoder.get_feature_names_out(CATEGORICAL_FEATURES).tolist()
    + abstract_feature_names
)

print("Train feature matrix shape:", X_train.shape)
print("Test feature matrix shape:", X_test.shape)
print("Number of positive labels:", int(y_binary.sum()))
print("Number of feature names:", len(feature_names))

In [ ]:
# Confirm the leakage-safe train/test split created above.
print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Train positives:", int(y_train.sum()))
print("Test positives:", int(y_test.sum()))

## Train Decision Trees at Several Depths

In [ ]:
def train_decision_tree(
    X_train,
    y_train: np.ndarray,
    max_depth: int | None = 5,
    random_state: int = 42,
    class_weight: str | None = "balanced",
    min_samples_leaf: int = 200,
    min_samples_split: int = 400,
):
    """
    Train and return a DecisionTreeClassifier.

    This mirrors the homework function, with class_weight='balanced' added because
    the highly-cited class is much smaller than the not-highly-cited class.

    The min_samples_leaf and min_samples_split settings regularize the tree so it
    cannot make splits from tiny, noisy groups of papers.
    """
    model = DecisionTreeClassifier(
        max_depth=max_depth,
        random_state=random_state,
        class_weight=class_weight,
        min_samples_leaf=min_samples_leaf,
        min_samples_split=min_samples_split,
    )
    model.fit(X_train, y_train)
    return model

In [ ]:
# Train at several depths to observe the effect with regularization.
for depth in [2, 3, 5, 8, 12, None]:
    model = train_decision_tree(X_train, y_train, max_depth=depth)
    label = str(depth) if depth is not None else "unlimited"
    print(
        f"max_depth={label:>9} | "
        f"actual_depth={model.tree_.max_depth:>3} | "
        f"leaves={model.tree_.n_leaves:>4} | "
        f"train_accuracy={model.score(X_train, y_train):.3f} | "
        f"min_samples_leaf={model.min_samples_leaf}"
    )

## Evaluate Classifier

In [ ]:
def evaluate_classifier(model, X_test, y_test: np.ndarray) -> dict:
    """
    Evaluate a classifier and return performance metrics.

    Returns accuracy, precision, recall, F1, and confusion-matrix counts.
    """
    y_pred = model.predict(X_test)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred, labels=[0, 1]).ravel()

    return {
        "accuracy": float(accuracy_score(y_test, y_pred)),
        "precision": float(precision_score(y_test, y_pred, zero_division=0)),
        "recall": float(recall_score(y_test, y_pred, zero_division=0)),
        "f1": float(f1_score(y_test, y_pred, zero_division=0)),
        "true_negatives": int(tn),
        "false_positives": int(fp),
        "false_negatives": int(fn),
        "true_positives": int(tp),
    }

In [ ]:
depth_results = []

for depth in [2, 3, 5, 8, 12, None]:
    model = train_decision_tree(X_train, y_train, max_depth=depth)
    metrics = evaluate_classifier(model, X_test, y_test)
    depth_results.append({
        "max_depth": str(depth) if depth is not None else "unlimited",
        "actual_depth": model.tree_.max_depth,
        "leaves": model.tree_.n_leaves,
        "min_samples_leaf": model.min_samples_leaf,
        "min_samples_split": model.min_samples_split,
        "train_accuracy": round(float(model.score(X_train, y_train)), 3),
        "test_accuracy": round(metrics["accuracy"], 3),
        "precision": round(metrics["precision"], 3),
        "recall": round(metrics["recall"], 3),
        "f1": round(metrics["f1"], 3),
        "false_positives": metrics["false_positives"],
        "false_negatives": metrics["false_negatives"],
    })

depth_results_df = pl.DataFrame(depth_results)
depth_results_df

## Visualize the Decision Tree

In [ ]:
# Use a regularized medium-depth model for the visualization: complex enough to learn
# patterns, but still less noisy than the unregularized tree.
dt_model = train_decision_tree(X_train, y_train, max_depth=5)
dt_metrics = evaluate_classifier(dt_model, X_test, y_test)

print(f"Tree depth: {dt_model.tree_.max_depth}")
print(f"Number of leaves: {dt_model.tree_.n_leaves}")
print(dt_metrics)

In [ ]:
# Visualize the top 3 levels of the decision tree.
plt.figure(figsize=(24, 10))
plot_tree(
    dt_model,
    feature_names=feature_names,
    class_names=["not highly cited", "highly cited"],
    max_depth=3,
    filled=True,
    rounded=True,
    fontsize=8,
)
plt.title("Decision Tree for Highly Cited Papers (top 3 levels)")
plt.tight_layout()
plt.show()

### Comments on Performance

- Adding `min_samples_leaf=200` and `min_samples_split=400` regularizes the tree by preventing splits that only explain a tiny handful of papers.
- The target is very imbalanced: in this dataset, only about 5.9% of joined papers have at least one citation, so accuracy is not very informative by itself.
- Compared with the unregularized tree, the regularized tree usually makes fewer false-positive predictions and has fewer leaves, which makes it easier to interpret.
- The improvement is modest: regularization makes the classifier less chaotic, but precision and F1 are still low because citation prediction is noisy and the positive class is rare.
- Shallow trees such as max_depth 2 or 3 stay simple, while medium-depth trees such as max_depth 5 give the model more room to find patterns without fully memorizing the training set.
- For this dataset, the best tree is not necessarily the one with the highest accuracy. F1, precision, and recall are more useful because the positive class is rare.